# Scenario Comparison

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import pypsa

# Add the scripts/pypsa-de directory to the path
scripts_path = os.path.join(os.path.dirname(os.getcwd()), "scripts", "pypsa-de")
sys.path.append(scripts_path)

from flexibility_plots_scenario_comparison import (
    get_capacities,
    plot_capacity_comparison,
)
from flexibility_utils import tech_colors

kwargs = {
    "groupby": ["bus", "carrier"],
    "at_port": True,
    "nice_names": False,
}

In [ ]:
f = "/home/julian-geis/Documents/06_PhD/02Flexibility/runs/"
run = "20251024-flex-restrict-component-buildout-test"  # "20251023-flex-scenario-update" # "20251015-flex-scenario-update-27cl-3h" # "20251013_flex-scenarios-temporal-industry-load-27cl-3h"
scenarios = ["LowFlex50", "LowFlex75", "LowBatt50"]
years = [2035, 2045]

In [ ]:
networks = {}
for scenario in scenarios:
    for year in years:
        fn = f"{f}/{run}/{scenario}/networks/base_s_27__none_{year}.nc"
        networks[(scenario, year)] = pypsa.Network(fn)

In [ ]:
# load variables
results = {}
for scenario in scenarios:
    df = (
        pd.read_excel(
            f"{f}/{run}/{scenario}/ariadne/exported_variables_full.xlsx",
            index_col=list(range(5)),
            # index_col=["Model", "Scenario", "Region", "Variable", "Unit"],
            sheet_name="data",
        )
        .groupby(["Variable", "Unit"], dropna=False)
        .sum()
    ).round(5)
    results[scenario] = df

# Functions

In [ ]:
def plot_variable(variable_name, results, scenarios, unit=None):
    fig, ax = plt.subplots(figsize=(6, 4))

    for scenario in scenarios:
        data = results[scenario].loc[variable_name]

        if (unit is None) & (data.index[0] is not None):
            unit = data.index[0]
        elif unit is None:
            unit = "Unspecified unit"

        years = [int(col) for col in data.columns]
        ax.plot(
            years, data.values[0], marker="*", label=scenario, linewidth=2, markersize=8
        )

        # Set x-axis to show only the years in the data
        if years is not None:
            ax.set_xticks(years)

    ax.set_xlabel("Year")
    ax.set_ylabel(unit if unit and pd.notna(unit) else "Value")
    ax.set_title(variable_name)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Start

In [ ]:
# filter for vars
search = "Secondary Ener"
vars = results["MedFlex"].index.get_level_values("Variable").unique().to_list()
[v for v in vars if search in v]

## System cost

In [ ]:
# System costs
# plot_variable("Total Energy System Cost|Trade|Hydrogen", results, scenarios, unit="billion EUR/a")
# plot_variable("Total Energy System Cost|Trade|Electricity", results, scenarios, unit="billion EUR/a")
# plot_variable("Total Energy System Cost", results, scenarios, unit="billion EUR/a")
# plot_variable("Total Energy System Cost|Non Trade", results, scenarios, unit="billion EUR/a")
# plot_variable("Total Energy System Cost|Trade", results, scenarios, unit="billion EUR/a")
plot_variable("Total Energy System Cost|EU", results, scenarios, unit="billion EUR/a")

- system costs increase but not significantly
- import dependency is not clearly higher
- overall trade costs seem to be even lower for NoFlex atm (imports less hydrogen)

# Capacity

In [ ]:
plot_variable("Capacity|Electricity", results, scenarios)
# plot_variable("Capacity|Electricity|Biomass", results, scenarios)
# plot_variable("Capacity|Electricity|Coal", results, scenarios)
# plot_variable("Capacity|Electricity|Gas", results, scenarios)
# plot_variable("Capacity|Electricity|Solar", results, scenarios)
# plot_variable("Capacity|Electricity|Wind", results, scenarios)
# plot_variable("Capacity|Electricity|Storage Converter|Stationary Batteries", results, scenarios)
# plot_variable("Capacity|Electricity|Storage Reservoir|Stationary Batteries", results, scenarios)
# plot_variable("Capacity|Electricity|Storage Converter|Stationary Batteries", results, scenarios)
# plot_variable("Capacity|Electricity|Transmission", results, scenarios)

- capacities very similar; LowFlex has more solar
- Low Flex has highest batteries?? -> home batteries

In [ ]:
# plot_variable("Capacity|Electricity|Gas", results, scenarios)
plot_variable("Capacity|Electricity|Hydrogen", results, scenarios)

## Capacity Detailed

In [ ]:
for year in [2045]:
    n = networks[("MedFlex", year)]

    capacities_electricity = (
        n.statistics.optimal_capacity(
            bus_carrier=["AC", "low voltage"],
            **kwargs,
        )
        .filter(like="DE")
        .groupby("carrier")
        .sum()
        .drop(
            # transmission capacities
            ["AC", "DC", "electricity distribution grid"],
            errors="ignore",
        )
    )

    print(year)
    print(
        capacities_electricity[
            capacities_electricity.index.str.contains(
                "CCGT|OCGT|gas|H2 retrofit|H2 turbine|H2 CHP", case=False
            )
        ]
        / 1e3
    )
    print()

## Export capacities csv

In [ ]:
# f = "/home/julian-geis/repos/pypsa-de-flex/results/"
# f = "/home/julian-geis/Documents/06_PhD/02Flexibility/runs/"
# run = "20251031-flex-Base-1H"

# scenario = "Base"
# years = [2025, 2035, 2045]
# clusters = 27
# hours = 1

# networks = {}
# for year in years:
#     fn = f"{f}/{run}/{scenario}/networks/base_s_{clusters}__none_{year}.nc"
#     networks[year] = pypsa.Network(fn)

# kwargs = {
#     "groupby": ["name","bus", "carrier"],
#     "nice_names": False,
# }

# results_dict = {}

# for year in years:
#     n = networks[year]
#     capacities = (
#         n.statistics.optimal_capacity(**kwargs)
#         .filter(like="DE")
#     )
#     results_dict[year] = capacities

# results = pd.concat(results_dict, axis=1)
# results.columns = years  # Clean up column names
# results.to_csv(f + run + "/{scenario}_capacities_{clusters}cl_{hours}H.csv")

# Secondary Energy / Generation

In [ ]:
# plot_variable("Secondary Energy|Electricity|Gas", results, scenarios)
plot_variable("Secondary Energy|Electricity|Hydrogen", results, scenarios)

In [ ]:
for year in [2045]:
    n = networks[("MedFlex", year)]

    generation_electricity = (
        n.statistics.supply(
            bus_carrier=["AC", "low voltage"],
            **kwargs,
        )
        .filter(like="DE")
        .groupby("carrier")
        .sum()
        .drop(
            # transmission capacities
            ["AC", "DC", "electricity distribution grid"],
            errors="ignore",
        )
    )

    print(year)
    print(
        generation_electricity[
            generation_electricity.index.str.contains(
                "CCGT|OCGT|gas|H2 retrofit|H2 turbine|H2 CHP", case=False
            )
        ]
        / 1e6
    )
    print()

In [ ]:
# Convert networks from flat structure to nested structure to use repo function
# From: networks[(scenario, year)]
# To:   networks[scenario][year]

networks_nested = {}
for (scenario, year), network in networks.items():
    if scenario not in networks_nested:
        networks_nested[scenario] = {}
    networks_nested[scenario][year] = network

data = get_capacities(networks_nested, scenarios, [2035, 2045])
fig = plot_capacity_comparison(data, scenarios, [2035, 2045], tech_colors)

In [ ]:
plot_variable("Secondary Energy|Electricity|Curtailment", results, scenarios)

In [ ]:
# Generation
plot_variable("Secondary Energy|Electricity", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Biomass", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Coal", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Gas", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Solar", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Wind", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Curtailment", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Storage Losses", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Hydrogen", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Nuclear", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Hydro", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Waste", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Fossil", results, scenarios)
# plot_variable("Secondary Energy|Electricity|Non-Biomass Renewables", results, scenarios)

In [ ]:
## Trade

In [ ]:
# Trade
plot_variable("Trade|Primary Energy|Biomass|Net Imports", results, scenarios)
# plot_variable("Trade|Secondary Energy|Electricity|Gross Import|Volume", results, scenarios)
# plot_variable("Trade|Secondary Energy|Electricity|Volume", results, scenarios)
# plot_variable("Trade|Secondary Energy|Electricity|Volume", results, scenarios)
# plot_variable("Trade|Secondary Energy|Hydrogen|Volume", results, scenarios)
# plot_variable("Trade|Secondary Energy|Hydrogen|Volume", results, scenarios)

In [ ]:
assert 0

# Testing

## Dispatchable

In [ ]:
n = networks[("MediumFlex", 2035)]

cf_electricity = (
    n.statistics.capacity_factor(
        bus_carrier=["AC", "low voltage"],
        **kwargs,
    )
    .filter(like="DE")
    .groupby("carrier")
    .sum()
)

cf_electricity[
    cf_electricity.index.str.contains(
        "CCGT|OCGT|gas|H2 retrofit|H2 turbine|H2 CHP", case=False
    )
]
cf = generation_electricity / capacities_electricity / 1e3
cf[cf.index.str.contains("CCGT|OCGT|gas|H2 retrofit|H2 turbine|H2 CHP", case=False)]
disp_capa = (
    capacities_electricity[
        capacities_electricity.index.str.contains(
            "CCGT|OCGT|gas|H2 retrofit|H2 turbine|H2 CHP", case=False
        )
    ]
    / 1e3
)
disp_gen = (
    generation_electricity[
        generation_electricity.index.str.contains(
            "CCGT|OCGT|gas|H2 retrofit|H2 turbine|H2 CHP", case=False
        )
    ]
    / 1e6
)
disp_cf = disp_gen / (disp_capa * 2920) * 1e3
pd.concat(
    [disp_capa, disp_gen, disp_cf],
    axis=1,
    keys=["Capacity [GW]", "Generation [TWh]", "Capacity Factor"],
)

# load pkl
import pickle

fn = "/home/julian-geis/Documents/06_PhD/02Flexibility/runs/20251015-flex-scenario-update-27cl-3h/MediumFlex/flexibility/data/flexibility_contributions_raw.pkl"
with open(fn, "rb") as f:
    flexibility_data = pickle.load(f)
    flexibility_data[2045].loc["weekly"].loc[
        [
            "Supply_H2 OCGT",
            "Supply_CCGT",
            "Supply_urban central H2 CHP",
            "Supply_urban central gas CHP",
        ]
    ]

## Percentage Scenarios

In [ ]:
capacities = pd.read_csv(
    "/home/julian-geis/repos/pypsa-de-flex/flex-data/MediumFlex_capacities_27cl_3H.csv",
    index_col=[0, 1, 2],
)

In [ ]:
year = 2045
carrier = "urban central water pits discharger"  # "H2 Electrolysis"
n = networks[("LowFlex50", year)]
n.links[(n.links.carrier == carrier) & (n.links.index.str.startswith("DE"))][
    ["p_nom_opt", "p_nom_max"]
]  # .sum()

In [ ]:
capacities[str(year)].xs(carrier, level=2) / 2

In [ ]:
capacities[str(year)].xs(carrier, level=2).sum() / 1e3

In [ ]:
restrict_carriers = [
    "battery discharger",
    "rural air heat pump",
    "rural ground heat pump",
    "rural resistive heater",
    "urban central air heat pump",
    "urban central resistive heater",
    "urban decentral air heat pump",
    "urban decentral resistive heater",
    "H2 Electrolysis",
    "urban central water pits discharger",
    "urban central water tanks discharger",
    #   "urban decentral water pits discharger",
    "urban decentral water tanks discharger",
    "H2 Store",
]
capacities.groupby("carrier").sum().loc[restrict_carriers] / 1e3